## Example breast cancer dataset from sklearn for logistic regression practice

### EDA and data cleanliness checks

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

In [ ]:
df = load_breast_cancer(as_frame=True).frame
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.dtypes.value_counts()

In [ ]:
df["target"].value_counts()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

---

### Data preparation for training

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # important for classification
)


In [ ]:
pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

In [ ]:
param_grid = {
    "model__penalty": ["l2"],                  # keep simple/reliable
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["lbfgs"],                # supports L2, stable default
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="roc_auc",   # good default; could also use "f1" for imbalance
    cv=cv,
    n_jobs=-1
)

In [ ]:
grid.fit(X_train, y_train)

In [ ]:
print("Best CV ROC-AUC:", grid.best_score_)
print("Best params:", grid.best_params_)

In [ ]:
best_model = grid.best_estimator_

### Running inference on test set

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

In [ ]:
print("\nTEST ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification report:\n", classification_report(y_test, y_pred))

### Plotting metrics

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test,
    cmap="Blues"
)

In [ ]:
RocCurveDisplay.from_estimator(
    best_model,
    X_test,
    y_test
)